In [5]:
import torch
import torchcrop
from torchcrop.utils.io import make_constant_weather

weather = make_constant_weather(batch_size=2, n_days=150)
model = torchcrop.Lintul5Model()
output = model(weather, start_doy=60)

print(output.yield_)  # [B] final storage-organ biomass (g m-2)
print(output.lai.shape)  # [B, T+1] LAI trajectory
print(output.dvs.shape)  # [B, T+1] development stage trajectory

tensor([244.0944, 244.0944])
torch.Size([2, 151])
torch.Size([2, 151])


In [6]:
import torch
import torch.nn as nn
from torchcrop import Lintul5Model, CropParameters

# Wrap every parameter we want to learn as nn.Parameter. The model reads
# RUE from `ruetb(DVS)` (per-DVS table), so optimising the scalar
# `crop.rue` alone has no effect — make `ruetb` learnable instead (or
# `scale_factor_rue` if you want a single scalar multiplier).
crop = CropParameters().to(dtype=torch.float64)
crop.tsum1 = nn.Parameter(crop.tsum1.detach().clone())
crop.ruetb = nn.Parameter(crop.ruetb.detach().clone())

model = Lintul5Model(crop_params=crop).double()

# Every parameter we want to update must be in the optimizer list — that
# is also what `optimizer.zero_grad()` zeros each iteration.
optimizer = torch.optim.Adam([crop.tsum1, crop.ruetb], lr=1e-1)

# Per-batch target (out.yield_ has shape [B]); avoids a silent broadcast.
observed_yield = torch.tensor([1200.0, 1200.0], dtype=torch.float64)

for i in range(50):
    optimizer.zero_grad()
    out = model(weather.to(torch.float64), start_doy=60)
    loss = ((out.yield_ - observed_yield) ** 2).mean()
    loss.backward()
    optimizer.step()

    if i % 5 == 0:
        print(
            f"Iter {i:2d}: loss={loss.item():9.2f}  "
            f"yield={out.yield_.mean().item():7.2f}  "
            f"tsum1={crop.tsum1.item():6.2f}  "
            f"|∇tsum1|={crop.tsum1.grad.norm().item():.2e}  "
            f"|∇ruetb|={crop.ruetb.grad.norm().item():.2e}"
        )

Iter  0: loss=919990.85  yield= 240.84  tsum1=899.90  |∇tsum1|=4.74e+02  |∇ruetb|=3.02e+05
Iter  5: loss=717020.29  yield= 353.23  tsum1=899.40  |∇tsum1|=7.37e+02  |∇ruetb|=1.43e+05
Iter 10: loss=609267.85  yield= 419.44  tsum1=898.88  |∇tsum1|=8.92e+02  |∇ruetb|=2.31e+05
Iter 15: loss=483805.85  yield= 504.44  tsum1=898.36  |∇tsum1|=8.57e+02  |∇ruetb|=2.65e+05
Iter 20: loss=428719.75  yield= 545.23  tsum1=897.81  |∇tsum1|=1.17e+03  |∇ruetb|=1.79e+05
Iter 25: loss=347473.69  yield= 610.53  tsum1=897.26  |∇tsum1|=1.04e+03  |∇ruetb|=2.56e+05
Iter 30: loss=282320.58  yield= 668.66  tsum1=896.72  |∇tsum1|=7.23e+02  |∇ruetb|=2.35e+05
Iter 35: loss=254077.64  yield= 695.94  tsum1=896.23  |∇tsum1|=7.61e+02  |∇ruetb|=2.02e+05
Iter 40: loss=217708.81  yield= 733.41  tsum1=895.78  |∇tsum1|=7.63e+02  |∇ruetb|=1.61e+05
Iter 45: loss=191454.26  yield= 762.45  tsum1=895.33  |∇tsum1|=7.42e+02  |∇ruetb|=7.81e+04
